In [49]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt

In [50]:
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.mapping_migrations as mm

In [51]:
data_dir = project_root / "data/"
outputs_dir = project_root / "outputs/"
database_path = data_dir / "all_hiski_records.sqlite3"


In [52]:
FUZZY_SCORE_CUTOFF = 90
MIGRATION_KIND = "immigration"
EXPORT = ["html", "mp4"]

In [53]:
toponyms_with_coordinates = mm.load_toponym_lookup(
    outputs_dir / "json/db_toponyms_with_coordinates_extended.json"
)

## Load tables

In [54]:
emigrated_records, immigrated_records = mm.load_database_migration_tables(database_path)
parish_records = mm.load_parish_records(database_path)

## Resolve toponyms and coordinates

In [55]:
emigrations_with_counterpart = mm.attach_counterpart_toponyms(
    emigrated_records,
    "destination",
    toponyms_with_coordinates,
    FUZZY_SCORE_CUTOFF,
)
immigrations_with_counterpart = mm.attach_counterpart_toponyms(
    immigrated_records,
    "source",
    toponyms_with_coordinates,
    FUZZY_SCORE_CUTOFF
)

parish_coordinates = mm.build_parish_coordinates(
    parish_records, toponyms_with_coordinates
)
parish_names_not_in_dict = parish_coordinates.loc[
    ~parish_coordinates["parish_name"]
    .str.strip()
    .isin(toponyms_with_coordinates),
    "parish_name",
]
print(
    f"{len(parish_names_not_in_dict)} of {len(parish_coordinates)}",
    "parish names are not in the toponym dict",
)

  0%|          | 0/12375 [00:00<?, ?it/s]

  0%|          | 0/14358 [00:00<?, ?it/s]

1 of 522 parish names are not in the toponym dict


In [56]:
emigrations_located = mm.attach_coordinates(
    emigrations_with_counterpart, parish_coordinates
)
immigrations_located = mm.attach_coordinates(
    immigrations_with_counterpart, parish_coordinates
)

plottable_emigrations = mm.drop_records_missing_coordinates(
    emigrations_located
)
plottable_immigrations = mm.drop_records_missing_coordinates(
    immigrations_located
)
print(
    len(plottable_emigrations), "emigration rows and",
    len(plottable_immigrations), "immigration rows can be plotted",
)

135648 emigration rows and 165117 immigration rows can be plotted


In [57]:
migrations = mm.build_migrations(plottable_emigrations, plottable_immigrations)

## Interactive map

In [58]:
print(
    migrations[migrations["year"] > 0]["year"].min(),
    migrations[migrations["year"] > 0]["year"].max()
)

1722 1909


In [59]:
years = list(range(1720, 1921, 5))
plot_events = mm.prepare_plot_events(migrations, MIGRATION_KIND, region_radius_km=20, year_bins=list(range(1720, 1921, 5)))
yearly_routes = mm.build_yearly_routes(plot_events, 2)
#years = sorted(yearly_routes[mm.YEAR_BIN_COLUMN].unique())
print(f"{len(yearly_routes)} year-routes over {len(years)} years ({years[0]}-{years[-1]})")

471 places merged into 246 regions (radius 20 km)
9910 year-routes over 41 years (1720-1920)


In [60]:
yearly_routes["count"].describe()

count    9910.000000
mean       10.954087
std        19.089768
min         2.000000
25%         2.000000
50%         4.000000
75%        11.000000
max       534.000000
Name: count, dtype: float64

In [ ]:
migration_figure = mm.build_migration_figure(
    yearly_routes,
    MIGRATION_KIND,
    source_description="Hiski data",
    use_gradient=True
)
migration_figure.show()

In [ ]:
if MIGRATION_KIND in ("emigration", "immigration"):
    migration_type = MIGRATION_KIND
else:
    migration_type = "migration"

if "html" in EXPORT:
    export_dir = outputs_dir / f"html/{migration_type}_Hiski"
    Path.mkdir(export_dir, parents=True, exist_ok=True)
    (export_dir/f"{migration_type}_Hiski.html").write_text(migration_figure.to_html())

if "mp4" in EXPORT:
    export_dir = outputs_dir / f"mp4/{migration_type}_Hiski"
    frames_dir = (export_dir/"frames")
    Path.mkdir(export_dir, parents=True, exist_ok=True)
    image_paths = mm.render_frame_images(
        migration_figure, frames_dir, scale=2
    )
    mm.write_video(
        image_paths,
        export_dir/f"{migration_type}_figure_Hiski.mp4",
        frames_per_second=2,
    )
    for image in frames_dir.glob("*.png"):
        image.unlink()
    frames_dir.rmdir()

## Metrics

In [64]:
display("emigrated (destination)", mm.summarize_toponym_coverage(emigrations_located))
display("immigrated (source)", mm.summarize_toponym_coverage(immigrations_located))

'emigrated (destination)'

,count
rows,182137
empty place name,8016
exact match,134958
fuzzy match,6186
no match,32977
"matched, but no coordinates",3864
"matched, with coordinates",137280


'immigrated (source)'

,count
rows,212902
empty place name,8120
exact match,164700
fuzzy match,7080
no match,33002
"matched, but no coordinates",4065
"matched, with coordinates",167715


In [65]:
migrations["distance"] = mm.compute_distances_in_metres(migrations)
print(
    "mean:", round(migrations["distance"].mean() / 1000), "km\n"
    "median:", round(migrations["distance"].median() / 1000), "km\n"
    "mode:", round(migrations["distance"].mode()[0] / 1000), "km"
)

mean: 76 km
median: 34 km
mode: 19 km


In [71]:
unique_migrations = (
    migrations[["kind", "origin_name", "target_name", "distance"]]
    .drop_duplicates()
    .sort_values(by="distance", ascending=False)
)
unique_migrations.head(5)

,kind,origin_name,target_name,distance
289153,immigration,Utsjoki,Viipuri,1.028468e+06
209004,immigration,Tukholma,Kuusamo,9.312728e+05
64048,emigration,Kuusamo,Tukholma,9.312728e+05
102537,emigration,Sortavala,Inari,8.209965e+05
52285,emigration,Karunki,Tukholma,8.072779e+05


In [67]:
print(migrations["origin_name"].value_counts().head(10).to_string())

origin_name
Viipuri             11390
Sortavala            6152
Akaa                 5714
Helsinki             4503
Helsingin pitäjä     4243
Vihti                4165
Iisalmi              4040
Hämeenlinna          4010
Janakkala            3947
Tohmajärvi           3884


In [68]:
print(migrations["target_name"].value_counts().head(10).to_string())

target_name
Viipuri        20404
Nurmijärvi     11558
Sortavala       7842
Helsinki        6245
Hämeenlinna     5003
Turku           4863
Kuopio          4714
Tampere         4369
Kurkijoki       4284
Tohmajärvi      4101


In [69]:
mm.count_fuzzy_matches(emigrations_located).head(25)

counterpart_raw_name  counterpart_matched_key  counterpart_normalized_name  counterpart_match_score
Kyrckslätt            Kyrkslätt                Kirkkonummi                  94.736842                  153
Wehkalahti            Vehkalahti               Vehkalahti                   90.000000                  140
Karku                 Karkku                   Karkku                       90.909091                  129
Nurmjärvi             Nurmijärvi               Nurmijärvi                   94.736842                  126
Willmanstrand         Villmanstrand            Lappeenranta                 92.307692                  115
Bborg                 B:borg                   Pori                         90.909091                  107
St Carin              St. Carin                Kaarina                      94.117647                   97
Thomajärvi            Tohmajärvi               Tohmajärvi                   90.000000                   77
Jäskis                Jääskis               

In [70]:
for located_records, place_label in [
    (emigrations_located, "destination"),
    (immigrations_located, "source"),
]:
    unmatched_name_counts = mm.count_unmatched_names(located_records)
    print(f"\nTop unmatched {place_label} values:")
    print(unmatched_name_counts.head(10).to_string())
    print("total unmatched:", unmatched_name_counts.sum())


Top unmatched destination values:
counterpart_raw_name
kaup.      694
K:joki     461
S:mäki     391
Kaup.      381
-          311
Gardet     217
Mod:k:n    212
Rautio     209
K:slätt    186
Pielis     177
total unmatched: 32977

Top unmatched source values:
counterpart_raw_name
kaup.          791
K:joki         323
-              308
Kaup.          254
Rautio         225
Ilmola         213
Gammelby       208
Pielis         148
Walkeasaari    148
T:kyro         127
total unmatched: 33002
